# 08: Sample Data Generation

This notebook demonstrates the synthetic data generation capabilities of siege_utilities,
along with file hashing and remote download utilities:

1. **Sample Datasets** - Load bundled sample data
2. **Synthetic Population** - Generate fake demographic data
3. **Synthetic Businesses** - Generate fake business records
4. **Synthetic Housing** - Generate fake housing data
5. **Spatial Data Integration** - Combine synthetic data with geographic boundaries
6. **File Integrity & Hashing** - Compute and verify file hashes for data integrity
7. **Remote File Downloads** - Download files with retry logic and metadata inspection

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install faker requests
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import siege_utilities as su
from siege_utilities.data.sample_data import (
    generate_synthetic_population,
    generate_synthetic_businesses,
    generate_synthetic_housing,
    load_sample_data,
    list_available_datasets
)

import pandas as pd
import numpy as np

# Initialize logging
su.configure_shared_logging(level="INFO")
su.log_info("Sample data generation notebook initialized")

# --- Path configuration ---
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")

## 1. Available Sample Datasets

List and load bundled sample datasets.

In [ ]:
# List available datasets
# list_available_datasets() returns a Dict[str, Dict] of dataset metadata
datasets = list_available_datasets()
su.log_info(f"Found {len(datasets)} available sample datasets")
for name, info in datasets.items():
    desc = info.get('description', 'No description')
    su.log_info(f"  - {name}: {desc}")

In [ ]:
# Load a sample dataset (if available)
if datasets:
    first_name = next(iter(datasets))
    sample = load_sample_data(first_name)
    su.log_info(f"Loaded '{first_name}'")
    su.log_info(f"  Shape: {sample.shape}")
    su.log_info(f"  Columns: {list(sample.columns)[:5]}...")
else:
    su.log_warning("No bundled datasets available")

## 2. Generate Synthetic Population

Create fake demographic data for testing and development.

In [ ]:
# Generate synthetic population
population = generate_synthetic_population(size=100)

su.log_info(f"Generated {len(population)} synthetic population records")
su.log_info(f"Columns: {list(population.columns)}")
su.log_debug(f"Sample records:\n{population.head()}")
population.head()

In [ ]:
# Analyze the synthetic population
if 'age' in population.columns:
    su.log_info("Age distribution:")
    su.log_info(f"  Mean: {population['age'].mean():.1f}")
    su.log_info(f"  Min: {population['age'].min()}, Max: {population['age'].max()}")

if 'gender' in population.columns:
    su.log_info("Gender distribution:")
    for gender, count in population['gender'].value_counts().items():
        su.log_info(f"  {gender}: {count}")

In [ ]:
# Generate larger population for more realistic analysis
large_pop = generate_synthetic_population(size=1000)
su.log_info(f"Generated {len(large_pop)} people for analysis")

# Quick summary
memory_kb = large_pop.memory_usage(deep=True).sum() / 1024
su.log_info(f"Memory usage: {memory_kb:.1f} KB")

## 3. Generate Synthetic Businesses

Create fake business records for testing and development.

In [ ]:
# Generate synthetic businesses
businesses = generate_synthetic_businesses(business_count=50)

su.log_info(f"Generated {len(businesses)} synthetic businesses")
su.log_info(f"Columns: {list(businesses.columns)}")
businesses.head()

In [ ]:
# Analyze business types (if column exists)
type_cols = [c for c in businesses.columns if 'type' in c.lower() or 'industry' in c.lower()]
if type_cols:
    col = type_cols[0]
    su.log_info(f"Business distribution by {col}:")
    for btype, count in businesses[col].value_counts().head(10).items():
        su.log_info(f"  {btype}: {count}")

## 4. Generate Synthetic Housing

Create fake housing/property records.

In [ ]:
# Generate synthetic housing data
housing = generate_synthetic_housing(housing_count=100)

su.log_info(f"Generated {len(housing)} synthetic housing records")
su.log_info(f"Columns: {list(housing.columns)}")
housing.head()

In [ ]:
# Price distribution (if available)
price_cols = [c for c in housing.columns if 'price' in c.lower() or 'value' in c.lower()]
if price_cols:
    col = price_cols[0]
    su.log_info(f"Price distribution ({col}):")
    su.log_info(f"  Mean: ${housing[col].mean():,.0f}")
    su.log_info(f"  Median: ${housing[col].median():,.0f}")
    su.log_info(f"  Min: ${housing[col].min():,.0f}")
    su.log_info(f"  Max: ${housing[col].max():,.0f}")

## 5. Combining Synthetic Data with Spatial Data

Use synthetic data with geographic boundaries for testing.

In [ ]:
from siege_utilities.geo.spatial_data import get_census_boundaries
import matplotlib.pyplot as plt

# Get CA counties
counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')
ca_counties = counties[counties['statefp'] == '06'].copy()

# Assign random synthetic population counts to counties
np.random.seed(42)
ca_counties['synthetic_population'] = np.random.randint(10000, 1000000, size=len(ca_counties))

su.log_info(f"Assigned synthetic population to {len(ca_counties)} counties")
su.log_info(f"Total synthetic population: {ca_counties['synthetic_population'].sum():,}")

In [ ]:
# Visualize
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 12))
ca_counties.plot(
    column='synthetic_population',
    ax=ax,
    legend=True,
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.3
)
ax.set_title('Synthetic Population by County')
ax.axis('off')
plt.tight_layout()
pop_map_path = DATA_DIR / 'synthetic_population_map.png'
plt.savefig(str(pop_map_path), dpi=100, bbox_inches='tight')
su.log_info(f"Map saved to {pop_map_path}")
plt.close()

## 6. File Integrity & Hashing

Verify file integrity using cryptographic hashes. These functions are useful for:
- Validating downloaded files against known checksums
- Detecting file corruption or tampering
- Quick change detection for caching/pipeline reprocessing

In [ ]:
from siege_utilities.files.hashing import (
    generate_sha256_hash_for_file,
    get_file_hash,
    verify_file_integrity,
    get_quick_file_signature,
)

su.log_info("Hashing functions imported successfully")

In [ ]:
# Create a sample file in DATA_DIR and compute its SHA256 hash
sample_file = DATA_DIR / "hash_demo_sample.txt"
sample_file.write_text(
    "This is a sample file for demonstrating siege_utilities hashing.\n"
    "It contains multiple lines of text to create a realistic test case.\n"
    "File integrity verification is critical for data pipeline reliability.\n"
)
su.log_info(f"Created sample file: {sample_file} ({sample_file.stat().st_size} bytes)")

# Generate SHA256 hash using the dedicated function
sha256_hash = generate_sha256_hash_for_file(sample_file)
su.log_info(f"SHA256 hash: {sha256_hash}")

# Generate hashes with different algorithms using get_file_hash()
md5_hash = get_file_hash(sample_file, algorithm="md5")
sha1_hash = get_file_hash(sample_file, algorithm="sha1")
sha256_via_generic = get_file_hash(sample_file, algorithm="sha256")

su.log_info(f"MD5 hash:    {md5_hash}")
su.log_info(f"SHA1 hash:   {sha1_hash}")
su.log_info(f"SHA256 hash (via get_file_hash): {sha256_via_generic}")

# Confirm both SHA256 methods produce the same result
assert sha256_hash == sha256_via_generic, "SHA256 hashes should match"
su.log_info("Both SHA256 methods produce identical results (as expected)")

In [ ]:
# Verify file integrity by comparing against a known hash
# This is the core pattern for validating downloaded or transferred files

su.log_info("--- File Integrity Verification ---")

# Case 1: correct hash -> should pass
is_valid = verify_file_integrity(sample_file, sha256_hash, algorithm="sha256")
su.log_info(f"Verification with correct hash:   {is_valid}")

# Case 2: wrong hash -> should fail
wrong_hash = "0" * 64  # 64-char hex string that won't match
is_invalid = verify_file_integrity(sample_file, wrong_hash, algorithm="sha256")
su.log_info(f"Verification with incorrect hash: {is_invalid}")

# Case 3: verify with MD5 algorithm
is_valid_md5 = verify_file_integrity(sample_file, md5_hash, algorithm="md5")
su.log_info(f"Verification with correct MD5:    {is_valid_md5}")

assert is_valid is True, "Correct hash should verify"
assert is_invalid is False, "Incorrect hash should fail"
assert is_valid_md5 is True, "Correct MD5 should verify"
su.log_info("All integrity checks behaved as expected")

In [ ]:
# Quick file signatures for change detection
# get_quick_file_signature() is optimized for speed:
#   - Small files (<=1MB): full SHA256
#   - Large files (>1MB): first+last 64KB chunks + file stats
# Returns 'missing' for non-existent files

su.log_info("--- Quick File Signatures ---")

# Signature of our sample file
sig_original = get_quick_file_signature(sample_file)
su.log_info(f"Original signature: {sig_original}")

# Modify the file and show the signature changes
sample_file.write_text(
    "This file has been modified to demonstrate change detection.\n"
)
sig_modified = get_quick_file_signature(sample_file)
su.log_info(f"Modified signature: {sig_modified}")

assert sig_original != sig_modified, "Signatures should differ after modification"
su.log_info(f"Signatures differ: {sig_original != sig_modified} (change detected)")

# Signature of a non-existent file
sig_missing = get_quick_file_signature(DATA_DIR / "does_not_exist.txt")
su.log_info(f"Missing file signature: '{sig_missing}'")

# Clean up demo file
sample_file.unlink(missing_ok=True)
su.log_info("Cleaned up hash demo file")

## 7. Remote File Downloads

Download files from URLs with progress tracking, retry logic, and metadata inspection.
These utilities handle SSL verification, timeouts, and chunked streaming for large files.

In [ ]:
from siege_utilities.files.remote import (
    download_file,
    download_file_with_retry,
    get_file_info,
    is_downloadable,
)

su.log_info("Remote file functions imported successfully")

In [ ]:
# Check whether URLs are downloadable
# is_downloadable() makes a HEAD request (and falls back to GET) to verify
# the URL serves file content

TEST_URL = "https://httpbin.org/robots.txt"
INVALID_URL = "https://httpbin.org/status/404"

try:
    can_download = is_downloadable(TEST_URL)
    su.log_info(f"is_downloadable('{TEST_URL}'): {can_download}")
except Exception as e:
    su.log_warning(f"Network error checking downloadability: {e}")
    can_download = None

try:
    cannot_download = is_downloadable(INVALID_URL)
    su.log_info(f"is_downloadable('{INVALID_URL}'): {cannot_download}")
except Exception as e:
    su.log_warning(f"Network error checking 404 URL: {e}")

In [ ]:
# Inspect remote file metadata without downloading
# get_file_info() returns a dict with: url, size, content_type, last_modified, etag

try:
    info = get_file_info(TEST_URL)
    if info:
        su.log_info("--- Remote File Info ---")
        for key, value in info.items():
            su.log_info(f"  {key}: {value}")
    else:
        su.log_warning("Could not retrieve file info (server may not support HEAD)")
except Exception as e:
    su.log_warning(f"Network error getting file info: {e}")

In [ ]:
# Download a small public file to DATA_DIR
# download_file() streams the content in chunks with a progress bar (if tqdm installed)
# Returns the local file path on success, or False on failure

download_dest = DATA_DIR / "robots.txt"

try:
    result = download_file(TEST_URL, download_dest, timeout=15)
    if result:
        dest_path = Path(result)
        su.log_info(f"Downloaded to: {dest_path}")
        su.log_info(f"File size: {dest_path.stat().st_size} bytes")

        # Verify the downloaded file with hashing
        dl_hash = generate_sha256_hash_for_file(dest_path)
        su.log_info(f"Downloaded file SHA256: {dl_hash}")

        # Clean up
        dest_path.unlink(missing_ok=True)
        su.log_info("Cleaned up downloaded demo file")
    else:
        su.log_warning("Download returned False (server may be unreachable)")
except Exception as e:
    su.log_warning(f"Download demo skipped due to network error: {e}")

In [ ]:
# download_file_with_retry() wraps download_file() with automatic retry logic
# Useful for flaky networks or large files that may time out
#
# Parameters:
#   max_retries: number of retry attempts (default 3)
#   retry_delay: seconds between retries (default 5)
#   **kwargs: forwarded to download_file()

retry_dest = DATA_DIR / "retry_demo.txt"

try:
    result = download_file_with_retry(
        TEST_URL,
        retry_dest,
        max_retries=2,
        retry_delay=1,
        timeout=15,
    )
    if result:
        dest_path = Path(result)
        su.log_info(f"Downloaded with retry to: {dest_path} ({dest_path.stat().st_size} bytes)")
        dest_path.unlink(missing_ok=True)
        su.log_info("Cleaned up retry demo file")
    else:
        su.log_warning("Download with retry returned False")
except Exception as e:
    su.log_warning(f"Retry download demo skipped due to network error: {e}")

## Summary

| Section | Function | Purpose |
|---------|----------|---------|
| **Sample Data** | `list_available_datasets()` | List bundled sample datasets |
| | `load_sample_data()` | Load a specific dataset |
| | `generate_synthetic_population()` | Create fake demographic data |
| | `generate_synthetic_businesses()` | Create fake business records |
| | `generate_synthetic_housing()` | Create fake housing data |
| **File Hashing** | `generate_sha256_hash_for_file()` | SHA256 hash with chunked reading |
| | `get_file_hash()` | Hash with configurable algorithm (sha256, md5, sha1, etc.) |
| | `verify_file_integrity()` | Compare file hash against expected value |
| | `get_quick_file_signature()` | Fast change detection (partial hash for large files) |
| **Remote Files** | `is_downloadable()` | Check if a URL serves downloadable content |
| | `get_file_info()` | Inspect remote file metadata (size, type, etag) |
| | `download_file()` | Stream download with progress bar |
| | `download_file_with_retry()` | Download with automatic retry on failure |

**Use Cases:**
- Testing data pipelines without real data
- Developing visualizations with synthetic datasets
- Validating downloaded files against known checksums
- Detecting file corruption or unauthorized changes
- Downloading remote data files with resilient retry logic
- Privacy-safe analytics development